In [1]:
#@title 3) Set paths (EDIT ME IF REQUIRED)
import os

# 🟨 STUDENT TODO: Set this to your Drive path
# Example: '/content/drive/MyDrive/data'
DATA_ROOT = "./content/"  # Local drive, running notebook locally.

TRAIN_DIR = os.path.join(DATA_ROOT, 'data', 'train')
SUBMISSION_DIR = os.path.join(DATA_ROOT, 'data', 'submission')

print('TRAIN_DIR:', TRAIN_DIR)
print('SUBMISSION_DIR:', SUBMISSION_DIR)
assert os.path.exists(TRAIN_DIR), 'Train directory not found'
assert os.path.exists(SUBMISSION_DIR), 'Submission directory not found'


TRAIN_DIR: ./content/data\train
SUBMISSION_DIR: ./content/data\submission


In [2]:
#@title 4) Imports
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score


# Load test dataset

In [7]:
labels_csv = os.path.join(SUBMISSION_DIR, 'metadata.csv')
audio_train_dir = os.path.join(SUBMISSION_DIR, 'audio')

df = pd.read_csv(labels_csv)
df['clip_id'] = df['clip_id'].astype(str)
print('Train rows:', len(df))
df.head()

Train rows: 1200


,clip_id
0,5-103415-A-2__clean
1,5-103416-A-2__clean
2,5-103418-A-2__clean
3,5-103420-A-2__clean
4,5-103421-A-2__clean


In [31]:
#@title ✅ Interactive Audio Player with Visualizations
import IPython.display as ipd
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import librosa.display

def play_audio_and_visualize(clip_id):
    row = df[df['clip_id'] == clip_id].iloc[0]
    path = os.path.join(audio_train_dir, f'{clip_id}.wav')
    y, sr = librosa.load(path, sr=16000)

    print(f"Clip ID: {clip_id} | Label: {row['clip_id']}")

    # Display audio player BEFORE the plots
    display(ipd.Audio(y, rate=sr))

    # Visualization
    fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))

    # Time Domain: Waveform
    librosa.display.waveshow(y, sr=sr, ax=ax[0])
    ax[0].set(title='Time Domain (Waveform)', xlabel='Time (s)', ylabel='Amplitude')

    # Frequency Domain: Spectrogram
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax[1])
    ax[1].set(title='Frequency Domain (Spectrogram)')

    plt.tight_layout()
    plt.show()

clip_selector = widgets.Dropdown(
    options=df['clip_id'].head(1000).tolist(),
    value=df['clip_id'].iloc[0],
    description='Train Clip:',
)

y = widgets.interact(play_audio_and_visualize, clip_id=clip_selector)

interactive(children=(Dropdown(description='Train Clip:', options=('5-103415-A-2__clean', '5-103416-A-2__clean…

In [95]:
def preprocess_audio(y, sr):
    from scipy.signal import butter, sosfilt
    """Basic preprocessing.

    🟨 STUDENT TODO: Improve this function.
    Ideas:
      - peak or RMS normalization
      - trim leading/trailing silence
      - fixed-length padding/truncation (e.g., 5s)
      - pre-emphasis filter
    """
    # Baseline: do nothing

    y, index = librosa.effects.trim(y, top_db=20)  # Trim away silence
    y = librosa.util.normalize(y)  # Peak normalisation (or amplification)
    y = librosa.util.fix_length(y, size=int(5.0 * sr))  # Pad to 5 seconds
    # y = librosa.effects.preemphasis(y, coef=0.97)

    # y = y + np.random.randn(len(y)).astype(np.float32) * 0.05

    # sos = butter(10, [300, 2500], btype='band', fs=sr, output='sos')
    # y = sosfilt(sos, y).astype(np.float32)

    # y = librosa.util.normalize(y)  # Peak normalisation (or amplification)

    return y

In [110]:
def play_audio_and_visualize(clip_id):
    row = df[df['clip_id'] == clip_id].iloc[0]
    path = os.path.join(audio_train_dir, f'{clip_id}.wav')
    y, sr = librosa.load(path, sr=16000)

    y = preprocess_audio(y, sr)
    
    print(f"Clip ID: {clip_id} | Label: {row['clip_id']}")

    # Display audio player BEFORE the plots
    display(ipd.Audio(y, rate=sr))

    # Visualization
    fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))

    # Time Domain: Waveform
    librosa.display.waveshow(y, sr=sr, ax=ax[0])
    ax[0].set(title='Time Domain (Waveform)', xlabel='Time (s)', ylabel='Amplitude')

    # Frequency Domain: Spectrogram
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax[1])
    ax[1].set(title='Frequency Domain (Spectrogram)')

    plt.tight_layout()
    plt.show()

clip_selector = widgets.Dropdown(
    options=df['clip_id'].head(1000).tolist(),
    value=df['clip_id'].iloc[0],
    description='Train Clip:',
)

x = widgets.interact(play_audio_and_visualize, clip_id=clip_selector)

interactive(children=(Dropdown(description='Train Clip:', options=('5-103415-A-2__clean', '5-103416-A-2__clean…

In [111]:
import pandas as pd

# 1. Import CSV
df = pd.read_csv('Pr_40_predictions.csv')

# 2. Extract base clip name and clip type
df['clip_type'] = df['clip_id'].str.extract(r'__(clean|noisy|bandlimited)$')
df['base_id'] = df['clip_id'].str.replace(r'__(clean|noisy|bandlimited)$', '', regex=True)

# 3. Pivot to display clean, noisy, bandlimited side-by-side
pivot = df.pivot_table(
    index='base_id',
    columns='clip_type',
    values='predicted_label',
    aggfunc='first'
)

# Reorder columns
pivot = pivot.reindex(columns=['clean', 'noisy', 'bandlimited'])

# Flag rows where predictions disagree across clip types
pivot['consistent'] = pivot.apply(
    lambda r: '✅' if r.nunique(dropna=True) == 1 else '❌', axis=1
)
pivot = pivot.reset_index(drop=True)

print(pivot.to_string())

clip_type             clean             noisy       bandlimited consistent
0                       pig          chainsaw               dog          ❌
1                   rooster           rooster           rooster          ✅
2                  laughing  drinking_sipping          laughing          ❌
3            glass_breaking         breathing          car_horn          ❌
4               crying_baby       crying_baby       crying_baby          ✅
5            glass_breaking  drinking_sipping          coughing          ❌
6                     siren             siren             siren          ✅
7                     siren             siren             siren          ✅
8                     siren             siren             siren          ✅
9                  coughing    glass_breaking          coughing          ❌
10                     wind              rain              wind          ❌
11                      pig               pig               dog          ❌
12                    sir

In [82]:
pivot

clip_type,clean,noisy,bandlimited,consistent
0,pig,chainsaw,pig,❌
1,rooster,rooster,rooster,✅
2,coughing,coughing,coughing,✅
3,car_horn,glass_breaking,car_horn,❌
4,crying_baby,crying_baby,pig,❌
...,...,...,...,...
395,hen,hen,hen,✅
396,washing_machine,vacuum_cleaner,washing_machine,❌
397,door_wood_knock,door_wood_knock,drinking_sipping,❌
398,sheep,sheep,airplane,❌


In [114]:
import pandas as pd

# 1. Load your predictions CSV
df = pd.read_csv('Pr_40_predictions.csv')
df.columns = df.columns.str.strip().str.lower()

# 2. Load ESC-50 ground truth
esc50 = pd.read_csv('esc50.csv')
esc50['lookup_key'] = esc50['src_file'].astype(str) + '-' + esc50['take'].astype(str)
lookup = esc50.set_index('lookup_key')['category'].to_dict()

# 3. Parse clip_id: e.g. "5-103415-A-2__noisy"
df['clip_type'] = df['clip_id'].str.extract(r'__(clean|noisy|bandlimited)$')
df['base_id'] = df['clip_id'].str.replace(r'__(clean|noisy|bandlimited)$', '', regex=True)

# Extract src_file and take: "5-103415-A-2" -> src=103415, take=A
df['src_file'] = df['base_id'].str.split('-').str[1]
df['take']     = df['base_id'].str.split('-').str[2]
df['lookup_key'] = df['src_file'] + '-' + df['take']

# 4. Map actual label from ESC-50
df['actual'] = df['lookup_key'].map(lookup)

# 5. Pivot predictions side-by-side
pivot = df.pivot_table(
    index=['base_id', 'actual'],
    columns='clip_type',
    values='predicted_label',
    aggfunc='first'
).reindex(columns=['clean', 'noisy', 'bandlimited'])

pivot.columns.name = None
pivot = pivot.reset_index()

# 6. Add consistency and correctness flags
pred_cols = ['clean', 'noisy', 'bandlimited']
pivot['consistent']   = pivot.apply(lambda r: '✅' if r[pred_cols].nunique(dropna=True) == 1 else '❌', axis=1)
pivot['correct_clean'] = pivot['clean'] == pivot['actual']
pivot['correct_noisy'] = pivot['noisy'] == pivot['actual']
pivot['correct_bandlimited'] = pivot['bandlimited'] == pivot['actual']

# 7. Display
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)
print(pivot.to_string(index=False))

# 8. Summary
print("\n--- Summary ---")
print(f"Total clips:      {len(pivot)}")
print(f"Consistent preds: {(pivot['consistent']=='✅').sum()} / {len(pivot)}")
print(f"Clean accuracy:   {pivot['correct_clean'].mean():.2%}")
print(f"Noisy accuracy:   {pivot['correct_noisy'].mean():.2%}")
print(f"Bandlimited accuracy:   {pivot['correct_bandlimited'].mean():.2%}")

      base_id           actual            clean            noisy      bandlimited consistent  correct_clean  correct_noisy  correct_bandlimited
 5-103415-A-2              pig              pig         chainsaw              dog          ❌           True          False                False
 5-103416-A-2              pig          rooster          rooster          rooster          ✅          False          False                False
 5-103418-A-2              pig         laughing         coughing         laughing          ❌          False          False                False
 5-103420-A-2              pig         car_horn        breathing         car_horn          ❌          False          False                False
 5-103421-A-2              pig      crying_baby      crying_baby      crying_baby          ✅          False          False                False
 5-103422-A-2              pig   glass_breaking  door_wood_knock   glass_breaking          ❌          False          False              